# Chapitre 1 — Télécharger les données (clé Kaggle)

Ce notebook **extrait le jeu de données RSNA** via l'API Kaggle. Prérequis des chapitres 2.5, 3, 4 et 5.

**Avant de lancer :**
1. Avoir accepté les règles de la compétition sur <https://www.kaggle.com/competitions/rsna-breast-cancer-detection/rules> (sinon erreur 403).
2. Identifiants Kaggle disponibles : `docker-run.sh` monte `~/.kaggle` en lecture seule dans le conteneur. Sinon, renseigner `KAGGLE_USERNAME` / `KAGGLE_KEY` (voir `.env.example`).

**Un seul réglage** plus bas, la variable `DOWNLOAD_MODE`, décide quoi télécharger : un petit sous-ensemble stratifié (défaut) ou le dataset complet.

In [ ]:
import os
# Vérifie les identifiants AVANT d'importer kaggle (l'import authentifie aussitôt).
os.environ.setdefault('KAGGLE_CONFIG_DIR', os.path.expanduser('~/.kaggle'))
have_json = os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json'))
have_env = bool(os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'))
assert have_json or have_env, (
    'Identifiants Kaggle absents : monte ~/.kaggle (kaggle.json) ou renseigne '
    'KAGGLE_USERNAME / KAGGLE_KEY (voir README).')
import kaggle  # déclenche l'authentification
print('Authentification Kaggle OK')

In [ ]:
import glob, zipfile

COMP = 'rsna-breast-cancer-detection'
DATA_DIR = os.path.expanduser('~/data/rsna')   # volume monté -> persistant
os.makedirs(DATA_DIR, exist_ok=True)

def _unzip_all(folder):
    """Décompresse puis supprime tous les .zip d'un dossier."""
    for z in glob.glob(os.path.join(folder, '*.zip')):
        print('décompression', os.path.basename(z))
        with zipfile.ZipFile(z) as zf:
            zf.extractall(folder)
        os.remove(z)

print('Cible :', DATA_DIR)

In [ ]:
# --- Labels seuls (léger) : de quoi explorer avant de tout télécharger ---
import pandas as pd
train_csv = os.path.join(DATA_DIR, 'train.csv')
if not os.path.exists(train_csv):
    kaggle.api.competition_download_file(COMP, 'train.csv', path=DATA_DIR, quiet=False)
    _unzip_all(DATA_DIR)
df = pd.read_csv(train_csv)
print('train.csv :', df.shape)
print('prévalence cancer :', round(df['cancer'].mean(), 4))
df.head()

## Choix : petit échantillon ou dataset complet ?

Modifie la variable `DOWNLOAD_MODE` ci-dessous, puis exécute la cellule de téléchargement. **Tu n'as pas à choisir quelles cellules lancer** — c'est la variable qui décide.

- `'sample'` — sous-ensemble **stratifié** de quelques patients (~50–80 Mo), idéal pour parcourir tout le cours rapidement ;
- `'full'` — **tout** le dataset DICOM (~314 Go, plusieurs heures, pic disque ~628 Go).

In [ ]:
# 'sample' : sous-ensemble stratifié (~5 patients) — défaut
# 'full'   : dataset RSNA complet (~314 Go)
DOWNLOAD_MODE = 'sample'      # <-- modifie ceci
N_PATIENTS = 4                # nombre de patients si mode 'sample'

## Téléchargement

En mode `'sample'`, on reprend la logique de `extract_download.py` : une ligne par patient (label = `max` du cancer), un `train_test_split` **stratifié** sur la prévalence, puis la garantie d'au moins un patient avec cancer et un sain. Les fichiers sont récupérés un par un via `competition_download_file`.

En mode `'full'`, on télécharge l'archive complète via `competition_download_files` (puis décompression).

In [ ]:
if DOWNLOAD_MODE == 'sample':
    from sklearn.model_selection import train_test_split
    SAMPLE_DIR = os.path.expanduser('~/data/rsna_sample')
    os.makedirs(os.path.join(SAMPLE_DIR, 'train_images'), exist_ok=True)

    # Patients ayant les 4 vues (examen complet) -> exploitables au ch2.5.
    NEED = {('L', 'CC'), ('L', 'MLO'), ('R', 'CC'), ('R', 'MLO')}
    views_by_pid = df.groupby('patient_id').apply(lambda g: set(zip(g['laterality'], g['view'])))
    complete = views_by_pid[views_by_pid.apply(NEED.issubset)].index
    grouped = df[df['patient_id'].isin(complete)].groupby('patient_id')['cancer'].max().reset_index()

    # Échantillon stratifié sur la prévalence du cancer (déterministe).
    selected, _ = train_test_split(
        grouped, train_size=N_PATIENTS, stratify=grouped['cancer'], random_state=42)
    for label in (1, 0):                       # garantir >=1 cancer et >=1 sain
        if not (selected['cancer'] == label).any():
            extra = grouped[grouped['cancer'] == label].sample(1, random_state=42)
            selected = pd.concat([selected, extra], ignore_index=True)

    pids = sorted(int(p) for p in selected['patient_id'])
    print('Patients (stratifiés) :', pids,
          '| cancer:', int(selected['cancer'].sum()),
          '| sains:', int((selected['cancer'] == 0).sum()))

    rows = df[df['patient_id'].isin(pids)]
    rows.to_csv(os.path.join(SAMPLE_DIR, 'train.csv'), index=False)
    for _, r in rows.iterrows():
        pid, iid = int(r['patient_id']), int(r['image_id'])
        dst = os.path.join(SAMPLE_DIR, 'train_images', str(pid))
        os.makedirs(dst, exist_ok=True)
        if os.path.exists(os.path.join(dst, f'{iid}.dcm')):
            continue                           # reprise : skip si déjà présent
        kaggle.api.competition_download_file(
            COMP, f'train_images/{pid}/{iid}.dcm', path=dst, quiet=True)
        _unzip_all(dst)
    n = len(glob.glob(os.path.join(SAMPLE_DIR, 'train_images', '**', '*.dcm'), recursive=True))
    print(f'{n} DICOM dans {SAMPLE_DIR} — prêts pour le chapitre 2.5 (mode sample).')

elif DOWNLOAD_MODE == 'full':
    if not os.path.isdir(os.path.join(DATA_DIR, 'train_images')):
        print('Téléchargement du dataset COMPLET (~314 Go)…')
        kaggle.api.competition_download_files(COMP, path=DATA_DIR, quiet=False)
        _unzip_all(DATA_DIR)
    n = len(glob.glob(os.path.join(DATA_DIR, 'train_images', '**', '*.dcm'), recursive=True))
    print(f'Dataset complet dans {DATA_DIR} ({n} DICOM) — prêt pour le chapitre 2.5 (mode full).')

else:
    raise ValueError("DOWNLOAD_MODE doit valoir 'sample' ou 'full'")